
# Version 2 — FAISS RAG
**Project:** AI Placement Preparation Assistant using RAG

V1 kept vectors in RAM. V2 uses **FAISS** — a fast similarity-search library by Meta — and **saves the index to disk**, so we embed only once.

**Colab secret needed:** `Gemini_API`

| # | Roadmap stage | Covered in this notebook |
|---|---|---|
| 1 | Document Loading | ✅ `PyPDFLoader` |
| 2 | Text Splitting / Chunking | ✅ `RecursiveCharacterTextSplitter` |
| 3 | Embeddings | ✅ Gemini embeddings |
| 4 | Vector Database | ✅ **FAISS** (saved + reloaded) |
| 5 | Retrieval | ✅ Retriever + scores |
| 6 | LLM / Gemini | ✅ Gemini chat model |
| 7 | Answer + Source/Page Citation | ✅ File + page number |
| 8 | Streamlit App | Later (`app.py`) |

### What is new compared to V1?
1. FAISS instead of the in-memory store
2. `save_local` / `load_local` → no need to re-embed
3. `similarity_search_with_score` → see *how close* each chunk is
4. A **score threshold** → refuse to answer when nothing is close enough
5. A **calibrated** threshold: the notebook measures your own questions and picks the value for you
6. **Quota-safe Gemini calls**: model fallback, cache, retry and a friendly message instead of a crash

> **How to run:** upload the 4 PDFs to Colab (left sidebar → Files), then run the cells **from top to bottom, one by one**. Step 10 sets `SCORE_THRESHOLD`; Step 14 tests the full pipeline. The chat loop at the end is switched off by default.


## Step 1 — Setup
`faiss-cpu` is the FAISS library (CPU version is enough for us).

In [ ]:
%pip install -q langchain langchain-community langchain-text-splitters langchain-google-genai faiss-cpu pypdf python-dotenv

## Step 2 — Gemini API key

In [ ]:
import os
from getpass import getpass

api_key = None

# 1) Google Colab: secret named Gemini_API  (left sidebar -> key icon)
try:
    from google.colab import userdata
    api_key = userdata.get("Gemini_API")
except Exception:
    pass

# 2) VS Code / local: a .env file in the project folder (GOOGLE_API_KEY=...) or an environment variable
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

# 3) Last option: type it (the text is hidden)
if not api_key:
    api_key = getpass("Enter your Gemini API key: ")

if not api_key:
    raise ValueError("No Gemini API key was provided.")

os.environ["GOOGLE_API_KEY"] = api_key
print("API key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))


## Step 3 — Check PDFs

In [ ]:
import os

from pathlib import Path

# Works in Google Colab (PDFs uploaded to /content) AND in VS Code (project folder with data/).
if Path("/content/hr_questions.pdf").exists():
    DATA_DIR = "/content"
    WORK_DIR = "/content"                      # where saved indexes go in Colab
else:
    _here = Path.cwd()
    _project = next((p for p in [_here, _here.parent] if (p / "data" / "hr_questions.pdf").exists()), _here)
    DATA_DIR = str(_project / "data")
    WORK_DIR = str(_project / "vectorstores")  # saved indexes go here (ignored by git)
os.makedirs(WORK_DIR, exist_ok=True)
print("PDF folder :", DATA_DIR)

PDF_FILES = [
    f"{DATA_DIR}/technical_interview_.pdf",
    f"{DATA_DIR}/resume_guidelines.pdf",
    f"{DATA_DIR}/hr_questions.pdf",
    f"{DATA_DIR}/aptitude.pdf",
]

missing = [f for f in PDF_FILES if not os.path.exists(f)]

if missing:
    print("❌ These files were not found:")
    for f in missing:
        print("   ", f)
    print("Upload them to Colab (left sidebar → Files) and run this cell again.")
else:
    print("✅ All 4 PDFs found")

## Step 4 — Load PDFs
📍 **Roadmap 1: Document Loading** — one Document per page (same as V1).

In [4]:
from langchain_community.document_loaders import PyPDFLoader

all_documents = []

for path in PDF_FILES:
    loader = PyPDFLoader(path)
    docs = loader.load()                      # one Document per page
    print(f"{os.path.basename(path):28s} -> {len(docs)} page(s)")
    all_documents.extend(docs)

print("\nTotal Documents (pages):", len(all_documents))

/tmp/ipykernel_1574/1712772264.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


technical_interview_.pdf     -> 5 page(s)
resume_guidelines.pdf        -> 3 page(s)
hr_questions.pdf             -> 3 page(s)
aptitude.pdf                 -> 4 page(s)

Total Documents (pages): 15


## Step 5 — Split into chunks
📍 **Roadmap 2: Chunking** — same idea as V1: 700 characters, 100 overlap.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,      # max characters in one chunk
    chunk_overlap=100,   # characters repeated between neighbouring chunks
)

chunks = text_splitter.split_documents(all_documents)

print("Pages  :", len(all_documents))
print("Chunks :", len(chunks))
print("\n--- Chunk 1 ---")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)

Pages  : 15
Chunks : 53

--- Chunk 1 ---
Technical Interview Preparation Notes: AI,
 Machine Learning and NLP
Section 1: Artificial Intelligence and Machine Learning Basics
What is Artificial Intelligence?
Artificial Intelligence is the field of building systems that perform tasks that normally need human
intelligence, such as understanding language, recognising images, making decisions and learning
from experience. Machine learning and deep learning are subsets of AI.
What is the difference between AI, Machine Learning and Deep Learning?
AI is the broad goal of making machines intelligent. Machine Learning is a subset of AI where
systems learn patterns from data instead of being explicitly programmed. Deep Learning is a subset

Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-08T11:57:06+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-08T11:57:06+00:00', 'subject': '(unspecified)', 'title': 'Techni

## Step 6 — Embeddings
📍 **Roadmap 3: Embeddings**

In [6]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Try it on one sentence
vector = embeddings.embed_query("What is overfitting?")
print("Vector length  :", len(vector))
print("First 5 numbers:", vector[:5])

Vector length  : 3072
First 5 numbers: [-0.006956061, -0.00035724926, 0.0017957747, -0.052621245, 0.009199971]


## Step 7 — Build the FAISS index
📍 **Roadmap 4: Vector Database**

**What is FAISS?** *Facebook AI Similarity Search.* A library that stores vectors and finds the closest ones very fast.

**Important difference from a full database:** FAISS focuses on the vector index. LangChain's wrapper keeps the chunk text + metadata next to it (in a docstore), so we still get text and page back.

`FAISS.from_documents(chunks, embeddings)` → embeds every chunk and builds the index.

In [7]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

print("✅ FAISS index created")
print("Vectors stored:", vectorstore.index.ntotal)

✅ FAISS index created
Vectors stored: 53


**Observe:** `ntotal` must equal the number of chunks.

## Step 8 — Save and reload the index
**Why:** embedding costs time and API calls. Save once, reuse later (also needed for Streamlit).

`allow_dangerous_deserialization=True` is required when loading because FAISS metadata is saved with Python *pickle*. It is safe **only for files you created yourself**. Never load an index from an unknown source.

In [ ]:
INDEX_DIR = os.path.join(WORK_DIR, "faiss_index")
vectorstore.save_local(INDEX_DIR)
print("Saved folder contents:", os.listdir(INDEX_DIR))

loaded_vectorstore = FAISS.load_local(
    INDEX_DIR,
    embeddings,
    allow_dangerous_deserialization=True,
)
print("✅ Reloaded. Vectors:", loaded_vectorstore.index.ntotal)

vectorstore = loaded_vectorstore

**Observe:** two files: `index.faiss` (the vectors) and `index.pkl` (the chunk text + metadata).

**Mentor may ask:** *Why two files?* → vectors and the text/metadata are stored separately.

## Step 9 — Retrieval with scores
📍 **Roadmap 5: Retrieval**

`similarity_search_with_score` returns `(chunk, score)`.

⚠️ In FAISS the score is a **distance (L2)**. **Smaller = more similar.** (Many beginners think bigger is better — here it is the opposite.)

In [9]:
question = "What is the difference between bagging and boosting?"

results = vectorstore.similarity_search_with_score(question, k=4)

for doc, score in results:
    print(f"score={score:.4f} | {os.path.basename(doc.metadata['source'])} | page {doc.metadata['page'] + 1}")
    print("   ", doc.page_content[:150].replace("\n", " "), "\n")

score=0.4940 | technical_interview_.pdf | page 2
    What is the difference between bagging and boosting? Bagging trains models independently in parallel on random samples and combines them, which reduce 

score=0.5221 | technical_interview_.pdf | page 2
    classification, commonly binary classification. How does a decision tree work? A decision tree splits the data repeatedly using feature conditions, ch 

score=0.7112 | technical_interview_.pdf | page 2
    testing on the remaining fold, and repeating this so every fold is used for testing. It gives a more reliable estimate of performance than a single tr 

score=0.7126 | technical_interview_.pdf | page 1
    What is the difference between classification and regression? Classification predicts a category or class, such as spam or not spam. Regression predic 



### Compare a related question with an unrelated one

In [10]:
for q in ["What is overfitting?", "Who is the prime minister of Japan?"]:
    best_doc, best_score = vectorstore.similarity_search_with_score(q, k=1)[0]
    print(f"{q:45s} -> best score = {best_score:.4f}")

What is overfitting?                          -> best score = 0.5285
Who is the prime minister of Japan?           -> best score = 0.9001


**Observe:** the unrelated question gets a **larger** (worse) distance. We use this gap to build a refusal rule.

> ⚠️ In the first run of this notebook, related questions scored about **0.4 – 0.55** and the unrelated one scored about **0.90**. So a good threshold lies between them (about 0.7). A threshold of 1.2 would be **too high**: it would never refuse anything, and every off-topic question would still be sent to Gemini and use up quota.

The next step measures several questions and calculates the threshold for you.

## Step 10 — Choose the score threshold (calibration)
**Idea:** put a few questions that *are* in the documents, and a few that are *not*, through the retriever. Look at the best distance of each. The threshold goes in the gap between the two groups.

This only uses the **embedding** API (a separate, larger quota than answer generation), so it does not use up your Gemini answer requests.

In [ ]:
# Edit these lists to match your own PDFs
IN_SCOPE = [          # questions that SHOULD be answered
    "What is overfitting?",
    "What is the STAR method?",
    "What is the formula for compound interest?",
    "What should a fresher resume include?",
    "What is the difference between bagging and boosting?",
    "How does linear regression work?",
]
OUT_OF_SCOPE = [      # questions that should be REFUSED
    "Who is the prime minister of Japan?",
    "What is the capital of Australia?",
    "Who won the FIFA World Cup in 2018?",
    "Give me a recipe for pizza dough.",
]


def best_distance(q):
    return vectorstore.similarity_search_with_score(q, k=1)[0][1]


in_scores = {q: best_distance(q) for q in IN_SCOPE}
out_scores = {q: best_distance(q) for q in OUT_OF_SCOPE}

print("IN-SCOPE (should be answered)")
for q, s in in_scores.items():
    print(f"  {s:.3f}  {q}")
print("\nOUT-OF-SCOPE (should be refused)")
for q, s in out_scores.items():
    print(f"  {s:.3f}  {q}")

worst_in = max(in_scores.values())       # the weakest match that must still be answered
best_out = min(out_scores.values())      # the strongest match that must be refused
print(f"\nLargest in-scope distance   : {worst_in:.3f}")
print(f"Smallest out-of-scope distance: {best_out:.3f}")

if worst_in < best_out:
    SCORE_THRESHOLD = round(float((worst_in + best_out) / 2), 2)      # middle of the gap
    print(f"\n✅ The two groups are separated. SCORE_THRESHOLD = {SCORE_THRESHOLD}")
else:
    SCORE_THRESHOLD = 0.7                                        # safe starting value
    print("\n⚠️ The groups overlap, so no perfect threshold exists. Using 0.7 as a starting point.")
    print("   Rephrase the overlapping questions or adjust SCORE_THRESHOLD by hand.")

# SCORE_THRESHOLD = 0.7   # <- remove the # to set the value yourself

## Step 11 — Gemini LLM (with automatic model fallback)
📍 **Roadmap stage 6: LLM**

**Why a list of models?** The Gemini *free tier* allows only a few requests **per model per day**. The quota is counted **separately for each model**, so we give the notebook a list. If model 1 returns a `429 RESOURCE_EXHAUSTED` error, LangChain's `.with_fallbacks()` automatically tries model 2, then model 3...

`max_retries=1` stops the library from silently retrying for a long time when the quota is already finished.

> The model names below depend on what your API key can use. Check **Google AI Studio → Rate limits** to see which models you have and how much quota is left. A wrong name is simply skipped.

In [ ]:
import warnings
from langchain_google_genai import ChatGoogleGenerativeAI

# Some models ignore "temperature" and print a warning on every call. It is harmless, so hide it.
warnings.filterwarnings("ignore", message=".*fixed sampling defaults.*")

# Free-tier quota is counted PER MODEL. If the first model is exhausted (429),
# the next one in the list is tried automatically.
MODEL_CANDIDATES = [
    "gemini-3.5-flash-lite",   # cost-efficient model (worked in the V3 run)
    "gemini-3.6-flash",         # backup model (separate free quota)
    # add more model names here only if they appear in Google AI Studio
]


def make_llm(model_name):
    return ChatGoogleGenerativeAI(
        model=model_name,
        max_retries=1,    # do not wait minutes retrying when the quota is gone
        timeout=60,
    )


llms = [make_llm(m) for m in MODEL_CANDIDATES]
llm = llms[0].with_fallbacks(llms[1:])     # model 1, then model 2, ...

print("Models, in the order they are tried:", MODEL_CANDIDATES)

## Step 12 — Prompt (anti-hallucination)
📍 **Roadmap stage 6: LLM (prompt part)**

A strict prompt: answer **only** from the retrieved context, otherwise reply with a fixed *not found* sentence.

`prompt | llm | StrOutputParser()` is a **chain**: the output of each piece goes into the next. The `|` symbol means "then".

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

NOT_FOUND = "I could not find the answer in the uploaded documents."

prompt = ChatPromptTemplate.from_template("""You are an AI Placement Preparation Assistant.

Answer the question using ONLY the context below.
If the answer is not in the context, reply exactly:
"{not_found}"
Do not use outside knowledge.

Context:
{context}

Question: {question}

Answer:""")

# prompt -> llm -> plain text
answer_chain = prompt | llm | StrOutputParser()

## Step 13 — Helper functions
`format_context` joins the chunks for the prompt. `format_sources` makes the *file - page* list (PyPDFLoader pages start at 0, humans start at 1).

In [14]:
def format_context(docs):
    """Join retrieved chunks into one text block for the prompt."""
    return "\n\n".join(d.page_content for d in docs)


def format_sources(docs):
    """Unique 'file - page' list. PyPDFLoader pages start at 0, humans start at 1."""
    seen, lines = set(), []
    for d in docs:
        name = os.path.basename(d.metadata.get("source", "unknown"))
        page = d.metadata.get("page", 0) + 1
        key = (name, page)
        if key not in seen:
            seen.add(key)
            lines.append(f"{name} - page {page}")
    return lines

## Step 13b — Handle quota errors (429) safely
**Problem:** when the free quota is finished, Gemini returns `429 RESOURCE_EXHAUSTED` and the notebook crashes with a long red traceback. Re-running cells also wastes the few requests you have.

**What this cell adds**
1. **Cache** → asking the same question again costs **0** requests.
2. **Wait and retry once** → only for *per-minute* limits (Gemini says how many seconds to wait). A *per-day* limit cannot be fixed by waiting, so we stop.
3. **Friendly message** → explains what happened and what to do, and still shows the **retrieved sources** (retrieval uses the embedding quota, which is separate from the answer-generation quota).

In [ ]:
import re
import time


class QuotaExhausted(Exception):
    """Raised when every Gemini model in MODEL_CANDIDATES has run out of quota."""

    def __init__(self, wait_seconds=None):
        super().__init__("Gemini quota exhausted")
        self.wait_seconds = wait_seconds


answer_cache = {}       # (question, k, threshold) -> answer  => repeated questions cost 0 requests
gemini_requests = 0     # generation requests sent in this notebook session


def is_quota_error(err):
    text = str(err)
    return "429" in text or "RESOURCE_EXHAUSTED" in text


def retry_delay_seconds(err):
    """Read the suggested wait (in seconds) from the error text. None if not given in seconds."""
    text = str(err)
    m = (re.search(r"retryDelay['\"]?:\s*['\"]?(\d+(?:\.\d+)?)s", text)
         or re.search(r"retry in (\d+(?:\.\d+)?)s", text))
    return float(m.group(1)) if m else None


def is_busy_error(err):
    """503 UNAVAILABLE = the model is busy for a moment. Waiting a few seconds usually fixes it."""
    text = str(err)
    return "503" in text or "UNAVAILABLE" in text or "high demand" in text


def generate_answer(question, context, max_wait=65):
    """Call Gemini. Retry once for short per-minute limits; raise QuotaExhausted otherwise."""
    global gemini_requests
    payload = {"context": context, "question": question, "not_found": NOT_FOUND}

    for attempt in (1, 2):
        try:
            gemini_requests += 1
            return answer_chain.invoke(payload)
        except Exception as err:
            if is_busy_error(err) and attempt == 1:
                print("⏳ Gemini is busy (503). Retrying once in 3 seconds...")
                time.sleep(3)
                continue
            if not is_quota_error(err):
                raise                                   # a different problem: show it
            wait = retry_delay_seconds(err)
            if attempt == 1 and wait is not None and wait <= max_wait:
                print(f"⏳ Per-minute limit reached. Waiting {wait:.0f}s, then retrying once...")
                time.sleep(wait + 1)
                continue
            raise QuotaExhausted(wait) from err         # per-day limit: waiting will not help


def show_quota_help(question, docs):
    """Friendly explanation + the chunks that retrieval found (retrieval does not need Gemini quota)."""
    print("Question:", question)
    print("\n⚠️  Could not generate an answer: the Gemini quota is finished for every")
    print("   model in MODEL_CANDIDATES (the free tier allows very few requests per day).")
    print("\n   What you can do:")
    print("   1. Add another model name to MODEL_CANDIDATES in Step 11, then re-run Step 11 and Step 12.")
    print("   2. Create a new API key in a NEW project (Google AI Studio) and update the Colab secret.")
    print("   3. Turn on billing for the project (much higher limits).")
    print("   4. Or wait for the daily reset.")
    print("\nRetrieval still worked. Most relevant chunks found:")
    for d in docs:
        name = os.path.basename(d.metadata.get("source", "unknown"))
        page = d.metadata.get("page", 0) + 1
        snippet = d.page_content[:150].replace("\n", " ")
        print(f" - {name} - page {page}: {snippet}...")

## Step 14 — RAG function with score threshold
📍 **Roadmap 5 + 6 + 7**

`SCORE_THRESHOLD` (calculated in Step 10): if even the best chunk is farther than this, we skip Gemini and say *not found*. This saves API calls and stops hallucination at the retrieval level.

> Gemini embedding distances depend on the data, so there is no universal value. That is why Step 10 measures yours.

The function also uses the cache and the safe `generate_answer()` from Step 13b, so a quota error prints a clear message instead of crashing.

In [ ]:
# Safety net: if Step 10 was skipped, use 0.7 so ask() never crashes
SCORE_THRESHOLD = globals().get("SCORE_THRESHOLD", 0.7)


def ask(question, k=4, show_scores=True):
    results = vectorstore.similarity_search_with_score(question, k=k)    # 1. retrieve
    best_score = results[0][1]

    if show_scores:
        print(f"(best distance = {best_score:.3f}, threshold = {SCORE_THRESHOLD})")

    if best_score > SCORE_THRESHOLD:                                     # 2. weak match -> refuse
        print("Question:", question)
        print("\nAnswer:\n", NOT_FOUND)
        print("\n(No Gemini request used: nothing in the documents is close enough.)")
        return

    docs = [doc for doc, score in results if score <= SCORE_THRESHOLD]
    cache_key = (question.strip().lower(), k, SCORE_THRESHOLD)

    if cache_key in answer_cache:                                        # same question asked before
        answer = answer_cache[cache_key]
        print("♻️  Answer taken from cache - no Gemini request used.\n")
    else:
        try:
            answer = generate_answer(question, format_context(docs))     # 3. generate
        except QuotaExhausted:
            show_quota_help(question, docs)
            return
        except Exception as err:                                         # any other error: short message
            print(f"❌ {type(err).__name__}: {str(err)[:300]}")
            return
        answer_cache[cache_key] = answer

    print("Question:", question)
    print("\nAnswer:\n", answer)
    if NOT_FOUND not in answer:                                          # 4. cite sources
        print("\nSources:")
        for s in format_sources(docs):
            print(" -", s)
    print(f"\n(Gemini requests sent this session: {gemini_requests})")


ask("What are the steps of a RAG pipeline?")

In [ ]:
ask("What is the formula for compound interest?")
print("\n" + "=" * 70 + "\n")
ask("How should I answer: Tell me about yourself?")
print("\n" + "=" * 70 + "\n")
ask("What is the capital of Australia?")

## Step 15 — Retriever object (needed for Streamlit later)
A retriever is a standard LangChain interface. Later versions plug into the same shape.

In [18]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(type(retriever).__name__)
print(len(retriever.invoke("What is a confusion matrix?")), "chunks retrieved")

VectorStoreRetriever
4 chunks retrieved


## Step 16 — Interactive chat
Type `exit` to stop.

In [19]:
# The chat loop waits for keyboard input, so it would block "Run all".
# Set RUN_CHAT = True only when you want to chat.
RUN_CHAT = False

while RUN_CHAT:
    q = input("\nYou: ")
    if q.strip().lower() in ["exit", "quit"]:
        print("Goodbye!")
        break
    ask(q)

print("Chat loop skipped (set RUN_CHAT = True above to use it)." if not RUN_CHAT else "")


Chat loop skipped (set RUN_CHAT = True above to use it).


## 🎓 Viva corner

**Q. What is FAISS?**

A library from Meta for very fast similarity search over vectors.

**Q. What does the FAISS score mean here?**

It is an L2 distance. Smaller means the chunk is closer in meaning to the question.

**Q. Why save the index?**

Embedding all chunks costs time and API calls. Saving lets us load the index instantly next time, and the Streamlit app can reuse it.

**Q. Why `allow_dangerous_deserialization=True`?**

The metadata file is a pickle. Pickle can run code when loaded, so we only load files that we created ourselves.

**Q. FAISS vs Chroma?**

FAISS is mainly the vector index. Chroma is a vector database that stores vectors + documents + metadata and supports metadata filtering and easy persistence.

**Q. What is the benefit of a score threshold?**

It refuses weak matches before calling the LLM, which reduces hallucination and API cost.

**Q. How did you choose the score threshold?**

I did not guess. I ran questions that are in the documents and questions that are not, looked at the best FAISS distance of each, and put the threshold in the gap between the two groups. Step 10 does this automatically.

**Q. What is the `429 RESOURCE_EXHAUSTED` error and how did you handle it?**

The Gemini free tier allows only a limited number of requests per model per day. When it is used up, the API answers with HTTP 429. We handled it with (1) a list of models with automatic fallback, (2) a cache so repeated questions cost nothing, (3) a short wait-and-retry for per-minute limits, and (4) a clear message that still shows the retrieved sources. The score threshold also helps because off-topic questions never reach Gemini. For a real deployment, billing should be enabled.

**Q. How do you stop hallucination in this version?**

Three layers: (1) the score threshold refuses weak matches before Gemini is called, (2) a strict prompt says to answer only from the retrieved context, otherwise reply with a fixed *not found* sentence, and (3) file and page citations let the user verify the answer.
